In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path
import sys
import re

import numpy as np
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/Legal_Sytem")
DATA_DIR = BASE_DIR / "01_Dataset/02_processed_data"
RETRIEVER_DIR = BASE_DIR / "02_Retriever"

SEARCH_DIRECTORIES = [
    RETRIEVER_DIR / "02_Baseline/BM25/output",
    RETRIEVER_DIR / "02_Baseline/BiEncoder/output",
    RETRIEVER_DIR / "02_Baseline/Dense Retriever/BiEncoder/Output",
    RETRIEVER_DIR / "03_Finetuning/output",
    RETRIEVER_DIR / "03_Finetuning/teacher",
]

OUTPUT_DIR = RETRIEVER_DIR / "05_Results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PAPER_TOP_K = [1, 5, 10, 15, 20, 25, 30]

REPORT_TOP_K = sorted(set(PAPER_TOP_K + [3]))

sys.path.append(str(RETRIEVER_DIR))

import importlib
import retriever_common
importlib.reload(retriever_common)
rc = retriever_common

print("Các mốc sẽ tính:", REPORT_TOP_K)

Các mốc sẽ tính: [1, 3, 5, 10, 15, 20, 25, 30]


## **01. Find ranking file**

In [ ]:
corpus = pd.read_parquet(DATA_DIR / 'corpus' / 'corpus.parquet')
query = pd.read_parquet(DATA_DIR / 'corpus' / 'queries.parquet')
qrels = pd.read_parquet(DATA_DIR / 'corpus' / 'qrels.parquet')

qrels_map_by_split = {
    split_name: rc.build_qrels_map(qrels[qrels['split'] == split_name])
    for split_name in ['train', 'val', 'test']
}

RANKING_PATTERN = re.compile(r'^(?:(?P<run>.+)__)?(?P<split>train|val|test)_ranking\.parquet$')

discovered = []

for directory in SEARCH_DIRECTORIES:
  if not directory.exists():
    continue

  for path in sorted(directory.glob('*_ranking.parquet')):
    match = RANKING_PATTERN.match(path.name)

    if not match:
      continue

    run_name = match.group('run')

    if run_name is None:
      run_name = 'BM25' if 'BM25' in str(directory) else directory.name

    discovered.append({
        'run_name': run_name,
        'split': match.group('split'),
        'path': path,
        'directory': directory.name,
    })

discovered_dataframe = pd.DataFrame(discovered)

if discovered_dataframe.empty:
  raise FileNotFoundError("Không tìm thấy file ranking nào. Kiểm tra SEARCH_DIRECTORIES.")

print(f"Tìm thấy {len(discovered_dataframe)} file ranking, "
      f"{discovered_dataframe['run_name'].nunique()} run khác nhau:\n")

display(discovered_dataframe.groupby(['run_name', 'split']).size().unstack(fill_value=0))

Tìm thấy 23 file ranking, 11 run khác nhau:



split,test,train,val
run_name,,,
BM25,1,1,1
BM25+bge-reranker-v2-m3__rerank50,0,0,1
Qwen3-Embedding-0.6B__marginmse-cross_encoder,1,0,1
Qwen3-Embedding-0.6B__truncated,1,0,1
Qwen3-Embedding-4B__marginmse-cross_encoder,1,0,1
Qwen3-Embedding-4B__truncated,1,0,1
bge-m3__marginmse-cross_encoder,1,0,1
bge-m3__truncated,1,0,1
multilingual-e5-large__chunked,1,1,1


### **2.1. Kiểm tra độ sâu**

In [11]:
depth_records = []

for record in discovered:
  ranking = pd.read_parquet(record['path'], columns=['query_id', 'rank'])

  minimum_depth = int(ranking.groupby('query_id')['rank'].max().min())

  depth_records.append({
      'run_name': record['run_name'],
      'split': record['split'],
      'min_depth': minimum_depth,
      'đủ_cho_k_max': minimum_depth >= max(REPORT_TOP_K),
  })

depth_dataframe = pd.DataFrame(depth_records)

shallow = depth_dataframe[~depth_dataframe['đủ_cho_k_max']]

if len(shallow):
  print(f"CẢNH BÁO: {len(shallow)} file nông hơn k lớn nhất ({max(REPORT_TOP_K)}).")
  print("P@k của chúng sẽ bị chặn ở độ sâu thật. Các mốc lớn hơn không dùng được.\n")
  display(shallow)
else:
  print(f"Mọi file đều sâu >= {max(REPORT_TOP_K)}. Tính được đủ các mốc.")

Mọi file đều sâu >= 30. Tính được đủ các mốc.


## **02. Tính lại các metric**

In [ ]:
metric_records = []
per_query_store = {}

for record in discovered:
  split_name = record['split']

  if split_name not in qrels_map_by_split:
    continue

  ranking = pd.read_parquet(record['path'])

  per_query = rc.evaluation(ranking, qrels_map_by_split[split_name], top_k_list=REPORT_TOP_K)

  if per_query.empty:
    print(f"Bỏ qua {record['run_name']} / {split_name}: không khớp query_id nào với qrels.")
    continue

  per_query_store[(record['run_name'], split_name)] = per_query

  summary = rc.summarize(per_query)

  row = {
      'run_name': record['run_name'],
      'split': split_name,
      'số_câu_hỏi': len(per_query),
  }

  for k in REPORT_TOP_K:
    row[f'P@{k}'] = round(summary[f'p@{k}'] * 100, 2)
    row[f'Recall@{k}'] = round(summary[f'recall@{k}'] * 100, 2)
    row[f'MRR@{k}'] = round(summary[f'mrr@{k}'] * 100, 2)
    row[f'nDCG@{k}'] = round(summary[f'ndcg@{k}'] * 100, 2)

  metric_records.append(row)

results = pd.DataFrame(metric_records)

print(f"Đã chấm lại {len(results)} kết quả.")

Đã chấm lại 23 kết quả.


In [13]:
results

,run_name,split,số_câu_hỏi,P@1,Recall@1,MRR@1,nDCG@1,P@3,Recall@3,MRR@3,...,MRR@20,nDCG@20,P@25,Recall@25,MRR@25,nDCG@25,P@30,Recall@30,MRR@30,nDCG@30
0,BM25,test,958,66.91,66.91,66.91,66.91,84.97,84.97,75.23,...,76.91,81.41,96.24,96.24,76.94,81.55,96.56,96.56,76.95,81.61
1,BM25,train,7697,69.13,69.05,69.13,69.13,86.54,86.50,76.97,...,78.56,82.88,97.14,97.13,78.59,83.01,97.58,97.57,78.60,83.11
2,BM25,val,933,71.28,71.28,71.28,71.28,87.03,87.03,78.30,...,79.83,83.86,96.68,96.68,79.84,83.88,97.53,97.53,79.87,84.06
3,Qwen3-Embedding-0.6B__truncated,test,958,56.47,56.47,56.47,56.47,77.66,77.66,65.90,...,68.47,74.63,95.72,95.72,68.53,74.90,96.35,96.35,68.55,75.03
4,Qwen3-Embedding-0.6B__truncated,val,933,57.23,57.23,57.23,57.23,76.96,76.96,65.95,...,68.79,74.86,95.61,95.61,68.84,75.12,96.14,96.14,68.86,75.23
5,Qwen3-Embedding-4B__truncated,test,958,63.99,63.99,63.99,63.99,83.40,83.40,72.69,...,74.77,79.93,97.18,97.18,74.81,80.11,97.49,97.49,74.82,80.18
6,Qwen3-Embedding-4B__truncated,val,933,64.63,64.63,64.63,64.63,82.64,82.64,72.65,...,74.87,80.05,97.00,97.00,74.89,80.12,97.43,97.43,74.90,80.21
7,bge-m3__truncated,test,958,61.38,61.38,61.38,61.38,80.79,80.79,70.11,...,72.53,78.15,97.18,97.18,72.57,78.35,97.91,97.91,72.60,78.50
8,bge-m3__truncated,val,933,62.06,62.06,62.06,62.06,81.24,81.24,70.61,...,72.89,78.36,96.78,96.78,72.93,78.55,97.21,97.21,72.95,78.64
9,multilingual-e5-large__chunked,test,958,58.25,58.25,58.25,58.25,79.96,79.96,68.02,...,70.90,77.10,97.49,97.49,70.93,77.26,97.81,97.81,70.94,77.32


### **2.1. Bảng theo từng tập**

In [14]:
# In du cot, hang cua df
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [16]:
paper_columns = ['run_name', 'số_câu_hỏi'] + [f'P@{k}' for k in PAPER_TOP_K] + [f'Recall@{k}' for k in PAPER_TOP_K] + [f'MRR@{k}' for k in PAPER_TOP_K] + [f'nDCG@{k}' for k in PAPER_TOP_K]

for split_name in ['val', 'test']:
  split_results = results[results['split'] == split_name]

  if split_results.empty:
    continue

  print(f"\n=== {split_name.upper()} ===")

  display(split_results[paper_columns].sort_values('P@1', ascending=False).reset_index(drop=True))


=== VAL ===


,run_name,số_câu_hỏi,P@1,P@5,P@10,P@15,P@20,P@25,P@30,Recall@1,Recall@5,Recall@10,Recall@15,Recall@20,Recall@25,Recall@30,MRR@1,MRR@5,MRR@10,MRR@15,MRR@20,MRR@25,MRR@30,nDCG@1,nDCG@5,nDCG@10,nDCG@15,nDCG@20,nDCG@25,nDCG@30
0,BM25+bge-reranker-v2-m3__rerank50,933,85.10,97.32,98.18,98.39,98.39,98.50,98.50,85.10,97.32,98.18,98.39,98.39,98.50,98.50,85.10,90.39,90.50,90.52,90.52,90.53,90.53,85.10,92.16,92.43,92.49,92.49,92.52,92.52
1,BM25,933,71.28,90.89,94.64,95.61,96.57,96.68,97.53,71.28,90.89,94.64,95.61,96.57,96.68,97.53,71.28,79.16,79.70,79.78,79.83,79.84,79.87,71.28,82.12,83.37,83.63,83.86,83.88,84.06
2,vietnamese-bi-encoder__marginmse-cross_encoder,933,69.77,95.50,97.75,98.39,98.93,99.04,99.25,69.77,95.50,97.75,98.39,98.93,99.04,99.25,69.77,80.74,81.05,81.10,81.13,81.14,81.14,69.77,84.50,85.22,85.40,85.52,85.55,85.59
3,bge-m3__marginmse-cross_encoder,933,66.88,90.57,93.46,94.75,95.82,96.68,97.21,66.88,90.57,93.46,94.75,95.82,96.68,97.21,66.88,76.42,76.83,76.93,76.99,77.03,77.05,66.88,79.98,80.94,81.28,81.54,81.73,81.84
4,Qwen3-Embedding-4B__truncated,933,64.63,88.32,93.46,95.71,96.68,97.00,97.43,64.63,88.32,93.46,95.71,96.68,97.00,97.43,64.63,73.95,74.64,74.82,74.87,74.89,74.90,64.63,77.56,79.23,79.82,80.05,80.12,80.21
5,Qwen3-Embedding-0.6B__marginmse-cross_encoder,933,62.70,88.32,92.18,94.32,95.39,96.36,96.89,62.70,88.32,92.18,94.32,95.39,96.36,96.89,62.70,72.68,73.24,73.41,73.46,73.51,73.53,62.70,76.60,77.89,78.46,78.71,78.92,79.03
6,bge-m3__truncated,933,62.06,86.60,92.93,94.75,95.93,96.78,97.21,62.06,86.60,92.93,94.75,95.93,96.78,97.21,62.06,71.83,72.69,72.83,72.89,72.93,72.95,62.06,75.54,77.61,78.08,78.36,78.55,78.64
7,multilingual-e5-large__chunked,933,58.84,87.35,94.53,96.57,97.11,97.64,97.96,58.84,87.35,94.53,96.57,97.11,97.64,97.96,58.84,70.61,71.58,71.74,71.77,71.79,71.80,58.84,74.85,77.18,77.72,77.85,77.96,78.03
8,Qwen3-Embedding-4B__marginmse-cross_encoder,933,57.34,82.64,88.64,91.00,93.14,94.11,95.50,57.34,82.64,88.64,91.00,93.14,94.11,95.50,57.34,67.51,68.29,68.48,68.59,68.64,68.69,57.34,71.32,73.24,73.86,74.37,74.58,74.86
9,Qwen3-Embedding-0.6B__truncated,933,57.23,83.60,90.68,93.14,94.43,95.61,96.14,57.23,83.60,90.68,93.14,94.43,95.61,96.14,57.23,67.52,68.51,68.71,68.79,68.84,68.86,57.23,71.56,73.90,74.56,74.86,75.12,75.23



=== TEST ===


,run_name,số_câu_hỏi,P@1,P@5,P@10,P@15,P@20,P@25,P@30,Recall@1,Recall@5,Recall@10,Recall@15,Recall@20,Recall@25,Recall@30,MRR@1,MRR@5,MRR@10,MRR@15,MRR@20,MRR@25,MRR@30,nDCG@1,nDCG@5,nDCG@10,nDCG@15,nDCG@20,nDCG@25,nDCG@30
0,vietnamese-bi-encoder__marginmse-cross_encoder,958,69.10,94.26,97.49,98.33,98.54,98.85,99.06,69.10,94.26,97.49,98.33,98.54,98.85,99.06,69.10,79.50,79.96,80.02,80.03,80.04,80.05,69.10,83.24,84.31,84.52,84.57,84.64,84.68
1,bge-m3__marginmse-cross_encoder,958,67.85,89.25,93.84,94.68,96.14,97.18,97.70,67.85,89.25,93.84,94.68,96.14,97.18,97.70,67.85,76.38,77.00,77.07,77.15,77.20,77.22,67.85,79.61,81.10,81.33,81.68,81.91,82.02
2,BM25,958,66.91,89.04,92.90,94.78,95.62,96.24,96.56,66.91,89.04,92.90,94.78,95.62,96.24,96.56,66.91,76.18,76.71,76.87,76.91,76.94,76.95,66.91,79.44,80.71,81.21,81.41,81.55,81.61
3,Qwen3-Embedding-4B__truncated,958,63.99,88.62,93.53,95.72,96.35,97.18,97.49,63.99,88.62,93.53,95.72,96.35,97.18,97.49,63.99,73.91,74.56,74.73,74.77,74.81,74.82,63.99,77.62,79.20,79.79,79.93,80.11,80.18
4,bge-m3__truncated,958,61.38,87.37,92.38,94.05,96.24,97.18,97.91,61.38,87.37,92.38,94.05,96.24,97.18,97.91,61.38,71.59,72.28,72.41,72.53,72.57,72.60,61.38,75.55,77.19,77.63,78.15,78.35,78.50
5,Qwen3-Embedding-0.6B__marginmse-cross_encoder,958,59.29,88.20,93.11,95.62,97.08,97.60,98.12,59.29,88.20,93.11,95.62,97.08,97.60,98.12,59.29,70.37,71.07,71.27,71.35,71.38,71.39,59.29,74.83,76.47,77.13,77.48,77.59,77.70
6,multilingual-e5-large__chunked,958,58.25,88.10,93.63,95.62,96.76,97.49,97.81,58.25,88.10,93.63,95.62,96.76,97.49,97.81,58.25,69.87,70.67,70.84,70.90,70.93,70.94,58.25,74.44,76.29,76.82,77.10,77.26,77.32
7,Qwen3-Embedding-4B__marginmse-cross_encoder,958,57.62,83.51,89.04,92.17,93.63,94.47,95.82,57.62,83.51,89.04,92.17,93.63,94.47,95.82,57.62,68.18,68.94,69.19,69.27,69.30,69.35,57.62,72.05,73.86,74.69,75.03,75.21,75.49
8,Qwen3-Embedding-0.6B__truncated,958,56.47,83.61,90.71,93.22,94.47,95.72,96.35,56.47,83.61,90.71,93.22,94.47,95.72,96.35,56.47,67.25,68.21,68.40,68.47,68.53,68.55,56.47,71.36,73.68,74.33,74.63,74.90,75.03
9,vietnamese-bi-encoder__chunked,958,51.98,77.87,85.59,89.14,91.44,93.11,94.26,51.98,77.87,85.59,89.14,91.44,93.11,94.26,51.98,62.10,63.12,63.40,63.53,63.60,63.64,51.98,66.06,68.55,69.49,70.03,70.39,70.63


In [17]:
results.to_csv(OUTPUT_DIR / 'all_results.csv', index=False, encoding='utf-8-sig')

## **03. Merge với kêt quả từ paper**

In [ ]:
paper_table = pd.DataFrame([
    {'run_name': '[paper] TF-IDF',              'P@1': 55.84, 'P@5': 86.66, 'P@10': 93.55, 'P@15': 95.80, 'P@20': 97.34, 'P@25': 98.16, 'P@30': 98.67},
    {'run_name': '[paper] BM25',                'P@1': 68.75, 'P@5': 91.39, 'P@10': 94.77, 'P@15': 96.93, 'P@20': 97.54, 'P@25': 98.05, 'P@30': 98.16},
    {'run_name': '[paper] LME (1)',             'P@1': 44.06, 'P@5': 74.49, 'P@10': 82.17, 'P@15': 87.60, 'P@20': 91.09, 'P@25': 92.32, 'P@30': 94.06},
    {'run_name': '[paper] LME (2)',             'P@1': 44.57, 'P@5': 73.98, 'P@10': 84.53, 'P@15': 88.42, 'P@20': 90.68, 'P@25': 92.32, 'P@30': 94.06},
    {'run_name': '[paper] LME (3)',             'P@1': 43.24, 'P@5': 72.34, 'P@10': 82.58, 'P@15': 86.17, 'P@20': 89.45, 'P@25': 91.09, 'P@30': 92.52},
    {'run_name': '[paper] LME (4)',             'P@1': 52.36, 'P@5': 76.85, 'P@10': 84.43, 'P@15': 88.22, 'P@20': 90.16, 'P@25': 91.19, 'P@30': 92.52},
    {'run_name': '[paper] Bi-Encoder',          'P@1': 61.27, 'P@5': 89.86, 'P@10': 94.67, 'P@15': 96.21, 'P@20': 97.03, 'P@25': 97.54, 'P@30': 97.85},
    {'run_name': '[paper] Weight-Ens TFIDF',    'P@1': 64.75, 'P@5': 91.08, 'P@10': 95.38, 'P@15': 96.82, 'P@20': 97.85, 'P@25': 98.16, 'P@30': 98.57},
    {'run_name': '[paper] Weight-Ens BM25',     'P@1': 72.13, 'P@5': 93.44, 'P@10': 96.72, 'P@15': 97.44, 'P@20': 98.05, 'P@25': 98.36, 'P@30': 98.46},
    {'run_name': '[paper] Mult-Ens TFIDF',      'P@1': 63.93, 'P@5': 90.06, 'P@10': 95.08, 'P@15': 96.82, 'P@20': 97.33, 'P@25': 98.05, 'P@30': 98.87},
    {'run_name': '[paper] Mult-Ens BM25',       'P@1': 72.74, 'P@5': 93.44, 'P@10': 96.52, 'P@15': 97.33, 'P@20': 98.05, 'P@25': 98.36, 'P@30': 98.66},
])

paper_table = paper_table.rename(columns={f'P@{k}': f'Acc@{k}' for k in PAPER_TOP_K})

paper_table['split'] = 'test (976 câu hỏi, số công bố)'
paper_table['số_câu_hỏi'] = 976

display(paper_table[['run_name', 'số_câu_hỏi'] + [f'Acc@{k}' for k in PAPER_TOP_K]])

In [ ]:
your_test_results = results[results['split'] == 'test'].copy()

combined = pd.concat([
    paper_table[['run_name', 'số_câu_hỏi'] + [f'Acc@{k}' for k in PAPER_TOP_K]],
    your_test_results[['run_name', 'số_câu_hỏi'] + [f'Acc@{k}' for k in PAPER_TOP_K]],
], ignore_index=True)

display(combined)

combined.to_csv(OUTPUT_DIR / 'final_table_test.csv', index=False, encoding='utf-8-sig')

results.to_csv(OUTPUT_DIR / 'all_results.csv', index=False, encoding='utf-8-sig')

print("Đã lưu:", OUTPUT_DIR / 'final_table_test.csv')

,run_name,số_câu_hỏi,Acc@1,Acc@5,Acc@10,Acc@15,Acc@20,Acc@25,Acc@30
0,[paper] TF-IDF,976,55.84,86.66,93.55,95.80,97.34,98.16,98.67
1,[paper] BM25,976,68.75,91.39,94.77,96.93,97.54,98.05,98.16
2,[paper] LME (1),976,44.06,74.49,82.17,87.60,91.09,92.32,94.06
3,[paper] LME (2),976,44.57,73.98,84.53,88.42,90.68,92.32,94.06
4,[paper] LME (3),976,43.24,72.34,82.58,86.17,89.45,91.09,92.52
5,[paper] LME (4),976,52.36,76.85,84.43,88.22,90.16,91.19,92.52
6,[paper] Bi-Encoder,976,61.27,89.86,94.67,96.21,97.03,97.54,97.85
7,[paper] Weight-Ens TFIDF,976,64.75,91.08,95.38,96.82,97.85,98.16,98.57
8,[paper] Weight-Ens BM25,976,72.13,93.44,96.72,97.44,98.05,98.36,98.46
9,[paper] Mult-Ens TFIDF,976,63.93,90.06,95.08,96.82,97.33,98.05,98.87


Đã lưu: /content/drive/MyDrive/Legal_Sytem/02_Retriever/05_Results/final_table_test.csv
